# Feature Engineering and Analysis

### Import Libraries

In [ ]:
import fiona
import geopandas as gpd
import numpy as np
import os
import pandas as pd

## 1. Load the Transshipment Datasets

In [ ]:
# CWD is the notebooks directory, as that is where Jupyter will run its kernel in.
TRANSSHIPMENT_DIR_FROM_CWD = "../data/raw/transshipment"
ENCOUNTER_EVENTS_FILENAME = "encounter-events-v20170717.csv"
LOITERING_EVENTS_FILENAME = "loitering-events-v20170717.csv"
VESSELS_FILENAME = "transshipment-vessels-v20170717.csv"

In [ ]:
transshipment_data_dir = os.path.join(os.getcwd(), TRANSSHIPMENT_DIR_FROM_CWD)

encounter_path = os.path.join(transshipment_data_dir, ENCOUNTER_EVENTS_FILENAME)
loitering_path = os.path.join(transshipment_data_dir, LOITERING_EVENTS_FILENAME)
transshipment_path = os.path.join(transshipment_data_dir, VESSELS_FILENAME)

encounter_df = pd.read_csv(encounter_path)
loitering_df = pd.read_csv(loitering_path)
transshipment_df = pd.read_csv(transshipment_path)


print("="*20 + " Encounter " + "="*20)
print("Shape: ",encounter_df.shape)
print("Example Row: ",encounter_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Loitering " + "="*20)
print("Shape: ",loitering_df.shape)
print("Example Row: ",loitering_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Transshipment " + "="*20)
print("Shape: ",transshipment_df.shape)
print("Example Row: ",transshipment_df.head(1))
print("="*20 + "=========" + "="*20)

## 2. Load the Global Marine Protected Area Data

In [ ]:
MPA_GDB_PATH_FROM_CWD = "../data/raw/global_mpa/WDPA_WDOECM_Apr2026_Public_marine.gdb"

In [ ]:
mpa_gdb_path = os.path.join(os.getcwd(), MPA_GDB_PATH_FROM_CWD)

fiona.listlayers(mpa_gdb_path)

In [ ]:
mpa_point_gdf = gpd.read_file(mpa_gdb_path, layer="WDPA_WDOECM_point_Apr2026_marine")

In [ ]:
mpa_location = mpa_point_gdf[['NAME_ENG', 'geometry']]

# Extract latitude and longitude
mpa_location['geometry'] = mpa_location['geometry'].explode(ignore_index=True)
mpa_location['lat'] = mpa_location['geometry'].y.astype(float)
mpa_location['long'] = mpa_location['geometry'].x.astype(float)
mpa_location.head()

## 3. Engineering

In [ ]:
loitering_loc = loitering_df[['transshipment_mmsi', 'starting_latitude', 'starting_longitude', 'ending_latitude', 'ending_longitude', 'total_event_duration', 'median_speed_knots', 'starting_timestamp']]

# Calculate the differences in lat and long from starting and ending positions
loitering_loc['diff_lat'] = loitering_loc['ending_latitude'] - loitering_loc['starting_latitude']
loitering_loc['diff_long'] = loitering_loc['ending_longitude'] - loitering_loc['starting_longitude']

# Determine if activity occured during night by correcting time (Local solar hour = UTC + longitude/15)
loitering_loc['starting_timestamp'] = pd.to_datetime(loitering_loc['starting_timestamp'])
loitering_loc['local_hour'] = (
    loitering_loc['starting_timestamp'].dt.hour +
    loitering_loc['starting_longitude'] / 15
) % 24

# Redefine is_night (True if between 6 PM and 6 AM)
loitering_loc['is_night_corrected'] = (
    (loitering_loc['local_hour'] >= 18) |
    (loitering_loc['local_hour'] < 6)
).astype(int)


loitering_loc['movement_mag'] = np.sqrt(
    loitering_loc['diff_lat']**2 + loitering_loc['diff_long']**2
)

#Calculate the distance to the nearest marine protected areas

loitering_nearest_mpa = loitering_loc.copy()

loitering_nearest_mpa['nearest_mpa_name'] = ""
loitering_nearest_mpa['nearest_mpa_lat'] = 0
loitering_nearest_mpa['nearest_mpa_long'] = 0
loitering_nearest_mpa['nearest_mpa_dist'] = 0


loit_coords = loitering_nearest_mpa[['starting_latitude', 'starting_longitude']].values
mpa_coords = mpa_location[['lat', 'long']].values


indices = [np.linalg.norm(mpa_coords - point, axis=1).argmin() for point in loit_coords]

nearest_mpas = mpa_location.iloc[indices]
loitering_nearest_mpa['nearest_mpa_name'] = nearest_mpas['NAME_ENG'].values
loitering_nearest_mpa['nearest_mpa_lat'] = nearest_mpas['lat'].values
loitering_nearest_mpa['nearest_mpa_long'] = nearest_mpas['long'].values

loitering_nearest_mpa['nearest_mpa_dist'] = np.sqrt(
    (loitering_nearest_mpa['starting_latitude'] - loitering_nearest_mpa['nearest_mpa_lat'])**2 +
    (loitering_nearest_mpa['starting_longitude'] - loitering_nearest_mpa['nearest_mpa_long'])**2
)

loitering_nearest_mpa.head()